In [33]:
# 입력 => 이미지, 좌표(x1, y1, x2, y2), 라벨
# 출력 => 좌표, 라벨

from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(224, 224, 3))

x = layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu")(inputs)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(filters=16, kernel_size=(3, 3), activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)

# 객체 박스 (회귀)
bbox = layers.Dense(units=4, activation="linear", name="bbox")(x)

# 객체 종류 (분류)
ccls = layers.Dense(units=5, activation="softmax", name="ccls")(x)

model = keras.Model(inputs, [bbox, ccls])
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 222, 222,  │        896 │ input_layer_12[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 111, 111,  │          0 │ conv2d_18[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 109, 109,  │      4,624 │ max_pooling2d_16… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ conv2d_19[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox (Dense)        │ (None, 4)         │         68 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ccls (Dense)        │ (None, 5)         │         85 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,673 (22.16 KB)

 Trainable params: 5,673 (22.16 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
! pip install keras-cv

In [35]:
import keras_cv
ciou_loss_fn = keras_cv.losses.CIoULoss(bounding_box_format="xyxy")

In [36]:
model.compile(
    optimizer="adam", 
    loss = {
        "bbox": "mse", 
        "ccls": "sparse_categorical_crossentropy"
    },
    metrics = {
        "bbox": "mae",
        "ccls": "accuracy"
    }
)

In [37]:
my_model = keras.models.clone_model(model)

In [38]:
my_model.compile(
    optimizer="adam", 
    loss = {
        "bbox": ciou_loss_fn, 
        "ccls": "sparse_categorical_crossentropy"
    },
    metrics = {
        "bbox": "mae",
        "ccls": "accuracy"
    }
)

In [39]:
import numpy as np

# 100장의 이미지 (랜덤하게 생성한 가상 데이터)
x = np.random.rand(100, 224, 224, 3).astype(np.float32)
print(f"x.shape: {x.shape}")

# bbox 
y_bbox = np.random.rand(100, 4).astype(np.float32)
print(f"y_bbox.shape: {y_bbox.shape}")

# ccls
y_ccls = np.random.randint(0, 5, size=(100,))
print(f"y_ccls.shape: {y_ccls.shape}")

x.shape: (100, 224, 224, 3)
y_bbox.shape: (100, 4)
y_ccls.shape: (100,)


In [40]:
history = model.fit(x, {"bbox": y_bbox, "ccls": y_ccls}, epochs=10, verbose=2)

Epoch 1/10
4/4 - 1s - 366ms/step - bbox_loss: 0.6581 - bbox_mae: 0.7541 - ccls_accuracy: 0.2500 - ccls_loss: 1.7072 - loss: 2.4289
Epoch 2/10
4/4 - 0s - 84ms/step - bbox_loss: 0.2603 - bbox_mae: 0.4324 - ccls_accuracy: 0.2700 - ccls_loss: 1.6207 - loss: 1.9217
Epoch 3/10
4/4 - 0s - 88ms/step - bbox_loss: 0.1577 - bbox_mae: 0.3327 - ccls_accuracy: 0.2700 - ccls_loss: 1.6704 - loss: 1.8240
Epoch 4/10
4/4 - 0s - 88ms/step - bbox_loss: 0.1396 - bbox_mae: 0.3055 - ccls_accuracy: 0.2700 - ccls_loss: 1.6208 - loss: 1.8101
Epoch 5/10
4/4 - 0s - 83ms/step - bbox_loss: 0.1159 - bbox_mae: 0.2881 - ccls_accuracy: 0.2700 - ccls_loss: 1.6440 - loss: 1.7965
Epoch 6/10
4/4 - 0s - 83ms/step - bbox_loss: 0.1171 - bbox_mae: 0.2776 - ccls_accuracy: 0.2700 - ccls_loss: 1.7575 - loss: 1.7873
Epoch 7/10
4/4 - 0s - 86ms/step - bbox_loss: 0.1099 - bbox_mae: 0.2782 - ccls_accuracy: 0.2700 - ccls_loss: 1.5994 - loss: 1.7792
Epoch 8/10
4/4 - 0s - 98ms/step - bbox_loss: 0.1091 - bbox_mae: 0.2801 - ccls_accuracy: 0

In [41]:
sample = np.random.rand(1, 224, 224, 3).astype(np.float32)
p_bbox, p_ccls = model.predict(sample)
p_bbox.shape, p_ccls.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step


((1, 4), (1, 5))

In [44]:
# 좌표 (4개의 실수), 종류

p_bbox[0], np.argmax(p_ccls[0], axis=0)

(array([0.2452877 , 0.34921062, 0.22824207, 0.45137015], dtype=float32),
 np.int64(3))